In [28]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)

# Corregimos las rutas para usar rutas absolutas dentro de Colab
RUTA_VENTAS = Path("/content/ventas.xlsx")
RUTA_PRODUCTO = Path("/content/productos.csv")
RUTA_SALIDA = Path("/content/ventas_consolidado.parquet")

print("Archivos disponibles:", RUTA_PRODUCTO.exists(), RUTA_VENTAS.exists())

Archivos disponibles: True True


In [29]:
# Cargamos los archivos en DataFrames de pandas primero
df_ventas = pd.read_excel(RUTA_VENTAS)
df_productos = pd.read_csv(RUTA_PRODUCTO)

# Ahora sí podemos usar .shape y .head()
print("Dimensiones de Ventas:", df_ventas.shape)
print("Dimensiones de Productos:", df_productos.shape)

df_ventas.head()

Dimensiones de Ventas: (3029, 7)
Dimensiones de Productos: (38, 5)


,ID_Venta,Fecha,ID_Cliente,ID_Producto,Cantidad,MÃ©todo_Pago,Estado
0,919,2024-01-31,10,25,5,1,Completa
1,947,2024-01-31,106,5,1,4,Completa
2,1317,2024-01-31,235,25,3,3,Completa
3,1607,2024-01-31,114,15,5,1,Completa
4,2038,2024-01-31,132,2,5,4,Completa


In [30]:
# Limpiar la columna 'Precio_Unitario' reemplazando comas por puntos y convirtiendo a float
df_productos['Precio_Unitario'] = df_productos['Precio_Unitario'].astype(str).str.replace(',', '.', regex=False)
df_productos['Precio_Unitario'] = df_productos['Precio_Unitario'].astype(float)

# Verificar el tipo de datos y mostrar los primeros registros
print(df_productos.dtypes)
display(df_productos.head())

ID_Producto          int64
Nombre_producto     object
Categoría           object
Precio_Unitario    float64
Stock                int64
dtype: object


,ID_Producto,Nombre_producto,Categoría,Precio_Unitario,Stock
0,1,Leche,Lácteos,12.24,3327
1,2,Yogur,Lácteos,5.21,3358
2,3,Queso cremoso,Lácteos,17.23,3167
3,4,Queso rallado,Lácteos,19.23,2099
4,5,Manteca,Lácteos,5.65,4929


In [31]:
# Renombrar la columna con problemas de codificación
df_ventas = df_ventas.rename(columns={'MÃ©todo_Pago': 'Método_Pago'})

# Mostrar las columnas actualizadas de df_ventas
print(df_ventas.columns)
display(df_ventas.head())

Index(['ID_Venta', 'Fecha', 'ID_Cliente', 'ID_Producto', 'Cantidad',
       'Método_Pago', 'Estado'],
      dtype='object')


,ID_Venta,Fecha,ID_Cliente,ID_Producto,Cantidad,Método_Pago,Estado
0,919,2024-01-31,10,25,5,1,Completa
1,947,2024-01-31,106,5,1,4,Completa
2,1317,2024-01-31,235,25,3,3,Completa
3,1607,2024-01-31,114,15,5,1,Completa
4,2038,2024-01-31,132,2,5,4,Completa


In [32]:
# Validación de df_ventas
print("=== VALIDACIÓN DE VENTAS ===")
print("\n--- Tipos de Datos (df_ventas.dtypes) ---")
print(df_ventas.dtypes)
print("\n--- Valores Nulos (df_ventas.isna().sum()) ---")
print(df_ventas.isna().sum())

# Validación de df_productos
print("\n\n=== VALIDACIÓN DE PRODUCTOS ===")
print("\n--- Tipos de Datos (df_productos.dtypes) ---")
print(df_productos.dtypes)
print("\n--- Valores Nulos (df_productos.isna().sum()) ---")
print(df_productos.isna().sum())

=== VALIDACIÓN DE VENTAS ===

--- Tipos de Datos (df_ventas.dtypes) ---
ID_Venta                int64
Fecha          datetime64[ns]
ID_Cliente              int64
ID_Producto             int64
Cantidad                int64
Método_Pago             int64
Estado                 object
dtype: object

--- Valores Nulos (df_ventas.isna().sum()) ---
ID_Venta       0
Fecha          0
ID_Cliente     0
ID_Producto    0
Cantidad       0
Método_Pago    0
Estado         0
dtype: int64


=== VALIDACIÓN DE PRODUCTOS ===

--- Tipos de Datos (df_productos.dtypes) ---
ID_Producto          int64
Nombre_producto     object
Categoría           object
Precio_Unitario    float64
Stock                int64
dtype: object

--- Valores Nulos (df_productos.isna().sum()) ---
ID_Producto        0
Nombre_producto    0
Categoría          0
Precio_Unitario    0
Stock              0
dtype: int64


In [33]:
# Realizar el merge (fusión) de df_ventas y df_productos por ID_Producto
df_consolidado = pd.merge(df_ventas, df_productos, on='ID_Producto', how='left')

# Verificar dimensiones del dataset resultante
print("Dimensiones del dataset consolidado:", df_consolidado.shape)

# Mostrar las primeras filas del DataFrame resultante
display(df_consolidado.head())

Dimensiones del dataset consolidado: (3029, 11)


,ID_Venta,Fecha,ID_Cliente,ID_Producto,Cantidad,Método_Pago,Estado,Nombre_producto,Categoría,Precio_Unitario,Stock
0,919,2024-01-31,10,25,5,1,Completa,Pizza congelada,Congelados,15.45,1640
1,947,2024-01-31,106,5,1,4,Completa,Manteca,Lácteos,5.65,4929
2,1317,2024-01-31,235,25,3,3,Completa,Pizza congelada,Congelados,15.45,1640
3,1607,2024-01-31,114,15,5,1,Completa,Medialunas,Panadería,3.51,4043
4,2038,2024-01-31,132,2,5,4,Completa,Yogur,Lácteos,5.21,3358


In [34]:
# Calcular la columna de venta total para cada registro
df_consolidado['Total_Venta'] = df_consolidado['Cantidad'] * df_consolidado['Precio_Unitario']

# Agrupar por Categoría y sumar el total de ventas
ventas_por_categoria = df_consolidado.groupby('Categoría')['Total_Venta'].sum().reset_index()

# Ordenar de mayor a menor total de ventas
ventas_por_categoria = ventas_por_categoria.sort_values(by='Total_Venta', ascending=False)

# Mostrar el resultado
display(ventas_por_categoria)

,Categoría,Total_Venta
1,Carnicería,28186.18
6,Lácteos,16179.93
2,Congelados,15034.41
7,Panadería,12800.70
0,Bebidas,11741.62
4,Frutas y Verduras,7905.38
5,Galletitas y Snacks,7165.18
3,Conservas,4933.96


In [35]:
# Exportar el DataFrame consolidado a formato Parquet
df_consolidado.to_parquet(RUTA_SALIDA, index=False)

# Confirmar la creación del archivo
print(f"Archivo guardado exitosamente en: {RUTA_SALIDA}")
print("¿Existe el archivo?:", RUTA_SALIDA.exists())

Archivo guardado exitosamente en: /content/ventas_consolidado.parquet
¿Existe el archivo?: True


In [36]:
# Convertir la columna Fecha a datetime de forma explícita con pd.to_datetime
df_consolidado['Fecha'] = pd.to_datetime(df_consolidado['Fecha'])

# Volver a exportar el DataFrame consolidado con la fecha asegurada
df_consolidado.to_parquet(RUTA_SALIDA, index=False)

# Verificar el tipo de dato final de la columna Fecha
print("Tipo de dato de 'Fecha':", df_consolidado['Fecha'].dtypes)
print("Archivo Parquet actualizado y guardado correctamente.")

Tipo de dato de 'Fecha': datetime64[ns]
Archivo Parquet actualizado y guardado correctamente.


In [37]:
# Exportar el DataFrame consolidado usando compresión Snappy
df_consolidado.to_parquet(RUTA_SALIDA, index=False, compression='snappy')

# Obtener el tamaño del archivo generado en Kilobytes (KB)
tamano_bytes = RUTA_SALIDA.stat().st_size
tamano_kb = tamano_bytes / 1024

print(f"Archivo guardado con éxito con compresión Snappy en: {RUTA_SALIDA}")
print(f"Tamaño del archivo final: {tamano_kb:.2f} KB ({tamano_bytes} bytes)")

Archivo guardado con éxito con compresión Snappy en: /content/ventas_consolidado.parquet
Tamaño del archivo final: 50.31 KB (51517 bytes)


### Verificación Exhaustiva de Valores Nulos en el Dataset Consolidado

In [38]:
# Verificar la cantidad y porcentaje de valores nulos por columna en el dataset consolidado
valores_nulos = df_consolidado.isna().sum()
porcentaje_nulos = (df_consolidado.isna().sum() / len(df_consolidado)) * 100

reporte_nulos = pd.DataFrame({
    'Nulos Cantidad': valores_nulos,
    'Porcentaje (%)': porcentaje_nulos
})

display(reporte_nulos)

,Nulos Cantidad,Porcentaje (%)
ID_Venta,0,0.0
Fecha,0,0.0
ID_Cliente,0,0.0
ID_Producto,0,0.0
Cantidad,0,0.0
Método_Pago,0,0.0
Estado,0,0.0
Nombre_producto,0,0.0
Categoría,0,0.0
Precio_Unitario,0,0.0


### Generación Automática del Archivo README.md
En esta sección crearemos el archivo de documentación utilizando la información recolectada durante el proceso de ETL.

In [39]:
contenido_readme = f"""# Proyecto de Consolidación y Transformación de Ventas

Este proyecto documenta el proceso de Extracción, Transformación y Carga (ETL) realizado sobre los conjuntos de datos de ventas y productos de la empresa.

## 1. Origen de los Datos
- **Ventas (`ventas.xlsx`):** Contiene el histórico de transacciones, fechas, clientes y métodos de pago.
- **Productos (`productos.csv`):** Catálogo de productos con sus precios unitarios, categorías y stock.

## 2. Decisiones de Ingeniería y Limpieza de Datos

### Manejo de Tipos de Datos Incompatibles
- **Precio Unitario:** La columna `Precio_Unitario` en el catálogo de productos original contenía valores con formato de texto usando comas como separador decimal (ej. "10,5"). Esto impedía realizar operaciones matemáticas directas. Se reemplazaron sistemáticamente las comas por puntos y se convirtió la columna a tipo numérico de punto flotante (`float64`).
- **Formato de Fechas:** Para garantizar una indexación temporal correcta y evitar interpretaciones inconsistentes entre sistemas, la columna `Fecha` se transformó explícitamente al tipo estandarizado `datetime64[ns]` utilizando `pd.to_datetime()`.
- **Codificación de Caracteres (Encoding):** Se corrigió la columna de ventas mal decodificada en la lectura (`MÃ©todo_Pago`), renombrándola a su forma correcta `Método_Pago`.

### Análisis de Valores Nulos
- **Decisión:** Al verificar inicialmente el conjunto de datos de Ventas y de Productos, se constató que **no existían valores nulos** en ninguna de las columnas clave antes de la fusión.
- **Validación Post-Merge:** Tras realizar la unión (`left join`), se realizó una verificación exhaustiva donde se confirmó un **0.0% de valores nulos**. Esto asegura la total integridad del catálogo de productos y de los registros transaccionales.

## 3. Análisis de Pérdida de Registros (Conciliación cuantitativa)
Durante todo el proceso de ETL **se perdieron 0 filas** (se conservaron el 100% de los registros, manteniendo exactamente las **3,029 filas** originales del set de ventas):
- **Por nulos:** No se detectaron valores faltantes que requirieran imputación o descarte.
- **Por duplicados:** Al tratarse de un histórico transaccional de ventas válidas, no se aplicaron eliminaciones de filas duplicadas para evitar la pérdida de transacciones legítimas repetidas.
- **Por IDs sin catálogo:** La unión fue perfecta. Todos los valores de `ID_Producto` presentes en la tabla de ventas existían dentro de la tabla del catálogo de productos (sin pérdidas por desajustes en la relación).
- **Por fechas inválidas:** No se presentaron inconsistencias de formato de fecha que provocaran descartes durante la conversión.

## 4. Resultados Clave

### Ventas Totales por Categoría
El análisis de rendimiento arrojó las siguientes ventas acumuladas por categoría de producto:

| Categoría | Total Ventas ($) |
| :--- | :--- |
| Carnicería | $28,186.18 |
| Lácteos | $16,179.93 |
| Congelados | $15,034.41 |
| Panadería | $12,800.70 |
| Bebidas | $11,741.62 |
| Frutas y Verduras | $7,905.38 |
| Galletitas y Snacks | $7,165.18 |
| Conservas | $4,933.96 |

## 5. Formato de Almacenamiento Final
El conjunto de datos consolidado se exportó bajo las siguientes especificaciones técnicas:
- **Formato:** Apache Parquet
- **Compresión:** Snappy (garantiza una excelente velocidad de lectura/escritura y optimización de espacio)
- **Ruta de salida:** `/content/ventas_consolidado.parquet`
- **Tamaño final en disco:** {tamano_kb:.2f} KB ({tamano_bytes} bytes)
"""

# Escribir el archivo README.md en el almacenamiento local de Colab
with open("/content/README.md", "w", encoding="utf-8") as f:
    f.write(contenido_readme)

print("¡Archivo README.md enriquecido y generado de manera exitosa en /content/README.md!")

¡Archivo README.md enriquecido y generado de manera exitosa en /content/README.md!
